In [ ]:
import polars as pl
import numpy as np

In [ ]:
wgs_dir = '/s/project/deeprvat_wgs/input_data/genebass_p1e6_genes_10kb/'

gl = pl.read_parquet(f'{wgs_dir}/gene_prot_coding_genebass.parquet')
gl

## Read Big annotations file and filter for set of variants

In [ ]:
ca = pl.scan_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/cadd_vep_annotations_processed_final_selected.parquet')

ca.collect_schema().names()

In [ ]:
ca_miss = ca.filter(
    pl.col('Gene').is_in(gl['region'].unique())
).filter(
    (pl.col('relative_cds_position_is_nan') == 0) |
    (pl.col('loftee_hc_is_nan') == 0) 
).collect()

ca_miss

In [ ]:
# sel_cols = [col for col in ca.collect_schema().names() if not col.endswith('is_nan')] + ['relative_cds_position_is_nan', 'loftee_hc_is_nan']

# ca_miss = ca_miss.select(sel_cols)
# ca_miss

## Read the VEP file to get amino acid substitutions

In [ ]:
vep = pl.scan_parquet(f'{wgs_dir}/annotations_vep.parquet')
vep.head(10).collect()

In [ ]:
set(gl['region']) - set(vep.select('Gene').unique().collect()['Gene'])

In [ ]:
vep_mis = vep.filter(pl.col('Gene').is_in(gl['region'])).filter(pl.col('Amino_acids').str.contains('/'))

miss_cols = ['id', 'region', 'consequence', 'amino_acids', 'protein_position']
# 'SIFT', 'PolyPhen', 'am_pathogenicity', 'CADD_PHRED', 'CADD_RAW'

vep_mis = vep_mis.rename({
    '#CHROM': 'chrom',
    'POS': 'pos',
    'REF': 'ref',
    'ALT': 'alt',
    'Gene': 'region',
    'Consequence': 'consequence',
    'Amino_acids': 'amino_acids',
    'Protein_position': 'protein_position',
}).with_columns(
    (pl.col('chrom') + ':' + pl.col('pos').cast(pl.Utf8) + ':' + pl.col('ref') + ':' + pl.col('alt')).alias('id')
).select(
    miss_cols
).collect()

vep_mis

## Merge this with missense annotations

In [ ]:
rap_miss = ca_miss.join(vep_mis, on=['id', 'region'], how='left')
rap_miss

In [ ]:
rap_miss.write_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass_genes_missense_lof_variants.parquet')

In [ ]:
rap_miss.filter(
    (pl.col('loftee_hc_is_nan') == 0) &
    (pl.col('amino_acids').is_null())
)